# SQL Lineage Explorer - Test Run
This notebook provides an interactive interface to explore lineage through your mined SQL codebase.

In [ ]:
import os
import json
from pathlib import Path
from loguru import logger

# 1. LLM Setup
from langchain_google_genai import ChatGoogleGenerativeAI

# Ensure your GOOGLE_API_KEY is set in environment
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    print("WARNING: GOOGLE_API_KEY not found in environment.")

llm_instance = ChatGoogleGenerativeAI(model="gemini-2.0-flash-exp", google_api_key=api_key)
print("LLM Instance Initialized.")

In [ ]:
# 2. Configuration
REGISTRY_PATH = r"path/to/registry.json" # Folder from separator
ENRICHED_DIR = r"path/to/enriched_output" # Folder from run_enricher.py

# Choose Mode: 'hybrid', 'graph', or 'tree'
EXPLORER_MODE = "hybrid" 

print(f"Mode selected: {EXPLORER_MODE}")

In [ ]:
# 3. Load Engines
from src.separator.registry import EntityRegistry
from src.enricher.core import GraphEnricher

# Load Registry
registry = EntityRegistry.load_from_file(REGISTRY_PATH)

# Load Enriched Lineage
enrich_path = Path(ENRICHED_DIR)
enricher = GraphEnricher(
    registry=registry,
    entities_json=str(enrich_path / "enriched_entities.json"),
    relationships_json=str(enrich_path / "enriched_relationships.json"),
    flows_json=str(enrich_path / "enriched_flows.json")
)

print(f"Engines ready. Registry count: {registry.count}")

In [ ]:
# 4. Initialize Harness
harness = None

if EXPLORER_MODE == "hybrid":
    from src.explorer.harness import ExplorerHarness
    harness = ExplorerHarness(enricher, llm=llm_instance)
elif EXPLORER_MODE == "graph":
    from src.explorer.graph_native.harness import GraphNativeHarness
    harness = GraphNativeHarness(enricher, llm=llm_instance)
elif EXPLORER_MODE == "tree":
    from src.explorer.tree_native.harness import TreeNativeHarness
    harness = TreeNativeHarness(registry, llm=llm_instance)

print(f"Harness for {EXPLORER_MODE} initialized.")

In [ ]:
# 5. Run Exploration
TARGET_TABLE = "EMPLOYEES"
TARGET_FIELD = "SALARY"

results = harness.run(TARGET_TABLE, TARGET_FIELD)
print("Exploration Complete.\n")

In [ ]:
# 6. Results Visualization
from IPython.display import Markdown, display

display(Markdown("## Discovery Report"))
display(Markdown(harness.load_wiki()))

# Save results locally
with open("discovery_results.json", "w") as f:
    json.dump(results, f, indent=2)